In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
import datetime
spark = SparkSession.builder.appName("UPI-transactions").getOrCreate()

In [0]:

username = dbutils.secrets.get(scope="credentials", key="RED_PANDA_USERNAME")
password = dbutils.secrets.get(scope="credentials", key="RED_PANDA_PASSWORD")
transactions_raw_binary = (spark.readStream
                   .format("kafka")
                   .option("kafka.bootstrap.servers", "d9s20tguj23020u21opg.any.ap-south-1.mpx.prd.cloud.redpanda.com:9092")
                   .option("subscribe", "UPI_Transactions")
                   .option("startingOffsets", "earliest")
                   .option("kafka.security.protocol", "SASL_SSL")
                   .option("kafka.sasl.jaas.config", f"kafkashaded.org.apache.kafka.common.security.scram.ScramLoginModule required username='{username}' password='{password}';")
                   .option("kafka.sasl.mechanism", "SCRAM-SHA-256")
                   .load()
                  )

In [0]:
transactions_string_convert = (transactions_raw_binary
                                .select(col("key").cast("string"),
                                        col("value").cast("string"),
                                        col("topic"),
                                        col("partition"),
                                        col("offset"),
                                        col("timestamp"),
                                        col("timestampType")
                                        )
                                )

In [0]:
query = (transactions_string_convert.writeStream
                   .format("delta")
                   .option("checkpointLocation", "/Volumes/workspace/upi_schema/checkpoints/BRONZE LAYER/bronze_kafka_unload/")
                   .outputMode("append")
                   .trigger(availableNow = True)
                   .queryName("transactions_bronze")
                   .toTable("workspace.upi_schema.upi_transactions_bronze")
        )

> # **Unloading Settlement File from S3**

In [0]:
settlement_schema = StructType([
    StructField("txn_id", StringType()),
    StructField("rrn", StringType()),
    StructField("settlement_amount", DecimalType(10,2)),
    StructField("settlement_status", StringType()),
    StructField("settlement_date", DateType()),
    StructField("settled_timestamp", TimestampType())
])

In [0]:
settled_read = (
                spark.readStream
                .format("cloudFiles")
                .option("cloudFiles.format","csv")
                .option("header","true")
                .schema(settlement_schema)
                .option("cloudFiles.schemaEvolutionMode","rescue")\
                .load("s3://settlement-file-databricks/")
                )

settled_read_with_runid = settled_read.withColumn("run_id", lit(dbutils.widgets.get("run_id")))

write_query = (
                settled_read_with_runid.writeStream
                .format("delta")
                .option("checkpointLocation","/Volumes/workspace/upi_schema/checkpoints/BRONZE LAYER/settlement_s3_unload/")
                .outputMode("append")
                .trigger(availableNow=True)
                .queryName("bronze_s3_unload")
                .toTable("workspace.upi_schema.bronze_settlement_data")
            )

> # Metrics

In [0]:
%sql

CREATE TABLE IF NOT EXISTS workspace.upi_schema.upi_pipeline_metrics (
    run_id STRING,
    run_date TIMESTAMP,
    txn_bronze_count LONG,
    settlement_bronze_count LONG,
    settlement_rescued_count LONG,
    txn_silver_count LONG,
    txn_silver_reject_count LONG,
    settlement_silver_count LONG,
    settlement_silver_rejected_count LONG,
    txn_rejection_rate DOUBLE,
    settlement_rejection_rate DOUBLE,
    gold_count LONG,
    gold_discrep_unsettled_success_count LONG,
    gold_discrep_stale_pending_count LONG,
    orphan_record_count LONG,
    discrepancy_rate DOUBLE
)

In [0]:
run_id = dbutils.widgets.get("run_id")
run_date = dbutils.widgets.get("run_date")
txn_bronze_count=0
query.awaitTermination()
for batch in query.recentProgress:
    txn_bronze_count += batch["sources"][0]["numInputRows"]
data =[run_id,run_date,txn_bronze_count]
cols=["run_id","run_date","txn_bronze_count"]
txn_br_count_df = spark.createDataFrame([data],cols)

txn_br_count_df.createOrReplaceTempView("txn_br_temp")

spark.sql("""
    MERGE INTO workspace.upi_schema.upi_pipeline_metrics t
    USING txn_br_temp s
    ON t.run_id = s.run_id
    WHEN MATCHED THEN UPDATE SET t.txn_bronze_count = s.txn_bronze_count
    WHEN NOT MATCHED THEN INSERT (run_id,run_date, txn_bronze_count) VALUES (s.run_id,s.run_date,s.txn_bronze_count)
""")

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
run_id = dbutils.widgets.get("run_id")
run_date = dbutils.widgets.get("run_date")
settlement_bronze_count=0
write_query.awaitTermination()
for batch in write_query.recentProgress:
    settlement_bronze_count += batch["sources"][0]["numInputRows"]
data =[run_id,run_date,settlement_bronze_count]
cols=["run_id","run_date","settlement_bronze_count"]

settlement_br_count_df = spark.createDataFrame([data],cols)
settlement_br_count_df.createOrReplaceTempView("settlement_br_temp")

spark.sql("""
    MERGE INTO workspace.upi_schema.upi_pipeline_metrics t
    USING settlement_br_temp s
    ON t.run_id = s.run_id
    WHEN MATCHED THEN UPDATE SET t.settlement_bronze_count = s.settlement_bronze_count
    WHEN NOT MATCHED THEN INSERT (run_id,run_date, settlement_bronze_count) VALUES (s.run_id,s.run_date, s.settlement_bronze_count)
""")

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
run_id = dbutils.widgets.get("run_id")
run_date = dbutils.widgets.get("run_date")
settlement_rescued_count=spark.read.table("workspace.upi_schema.bronze_settlement_data").filter(col("_rescued_data").isNotNull() & (col("run_id") == run_id)).count()
data =[run_id,run_date,settlement_rescued_count]
cols=["run_id","run_date","settlement_rescued_count"]

settlement_rescued_count_df = spark.createDataFrame([data],cols)

settlement_rescued_count_df.createOrReplaceTempView("settlement_rescued_temp")

spark.sql("""
    MERGE INTO workspace.upi_schema.upi_pipeline_metrics t
    USING settlement_rescued_temp s
    ON t.run_id = s.run_id
    WHEN MATCHED THEN UPDATE SET t.settlement_rescued_count = s.settlement_rescued_count
    WHEN NOT MATCHED THEN INSERT (run_id,run_date, settlement_rescued_count) VALUES (s.run_id,s.run_date,s.settlement_rescued_count)
""")

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]